# Spectral-gap-matched null graphs

**What this does.** It generates families of graphs that share a target spectral
gap to any tolerance you set, while differing as much as possible in local
triadic organisation. It is intended as the matched null model described as
missing in Deco et al., *Quantum-Like Dynamics in Whole-Brain Models of the
Human Connectome*, Adv. Sci. 2026, e77103: a design in which the spectral gap is
held fixed so that any change in model fit cannot be attributed to the gap.

**Why the triadic excess and not the clustering coefficient.** Both are local,
but they are not equally free of the gap. Measured over 600 pruned k-regular
graphs (n=40, k=20), the fraction of variation *not* explained by the spectral
gap is 62% for the per-node triadic excess over a configuration null and only
19% for the clustering coefficient. Matching on the gap therefore nearly matches
on clustering as well, which is why clustering cannot serve as the contrast.
Cell 4 reproduces that measurement so you do not have to take it on trust.

**Why it is possible at all.** The spectral gap is a functional of the spectrum.
The per-node triangle count is not: cospectral graphs with different per-node
triangle counts exist. So two graphs can agree on the gap and disagree on local
structure, and the search below simply finds them.

Runtime is about a minute on a free Colab CPU. Nothing is installed.

In [ ]:
#@title 1. Setup
import numpy as np, networkx as nx, pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, diags

EPS = 1e-12

def spectral_gap(A):
    """lambda_0 - lambda_1 of the connectivity matrix, as defined in the paper."""
    ev = np.sort(np.linalg.eigvalsh(A))[::-1]
    return float(ev[0] - ev[1])

def triadic_excess(A):
    """Per-node triangle count as an excess over the configuration null.

    E[t_i] = (s1_i^2 - s2_i) / 4m with s1, s2 the first two neighbour-degree
    moments. The +1 in the denominator keeps the ratio finite where the null
    expectation is near zero.
    """
    A = np.asarray(A, float)
    k = A.sum(1); m = k.sum() / 2
    if m <= 0: return np.zeros(len(k))
    tri = ((A @ A) * A).sum(1) / 2.0
    s1 = A @ k; s2 = A @ (k ** 2)
    E = (s1 ** 2 - s2) / (4 * m)
    return (tri - E) / (E + 1.0)

def summarise(G):
    n = G.number_of_nodes()
    A = nx.to_numpy_array(G, nodelist=sorted(G.nodes()))
    exc = triadic_excess(A)
    return dict(n=n, gap=spectral_gap(A), exc=float(np.median(exc)),
                clustering=nx.average_clustering(G),
                mean_degree=float(A.sum(1).mean())), A
print('ready')

In [ ]:
#@title 2. Choose your target { run: "auto" }
N_NODES = 40 #@param {type:"integer"}
K_REGULAR = 20 #@param {type:"integer"}
TARGET_GAP = 8.0 #@param {type:"number"}
GAP_TOLERANCE = 0.15 #@param {type:"number"}
GRAPHS_PER_FAMILY = 20 #@param {type:"integer"}
SEARCH_BUDGET = 4000 #@param {type:"integer"}
SEED = 0 #@param {type:"integer"}

print(f'Looking for graphs with spectral gap {TARGET_GAP} +/- {GAP_TOLERANCE},')
print(f'then splitting them into a HIGH and a LOW triadic-excess family of')
print(f'{GRAPHS_PER_FAMILY} graphs each.')
print()
print('If the search returns too few graphs, widen GAP_TOLERANCE or move')
print('TARGET_GAP towards the range your pruning naturally produces (cell 4')
print('shows that range).')

In [ ]:
#@title 3. Search
rng = np.random.default_rng(SEED)
hits = []
for it in range(SEARCH_BUDGET):
    p = rng.uniform(0.02, 0.90)
    G = nx.random_regular_graph(K_REGULAR, N_NODES, seed=int(rng.integers(1e9)))
    G.remove_edges_from([e for e in list(G.edges()) if rng.random() < p])
    if G.number_of_edges() == 0: continue
    G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    if G.number_of_nodes() < N_NODES * 0.5: continue
    G = nx.convert_node_labels_to_integers(G)
    s, A = summarise(G)
    if abs(s['gap'] - TARGET_GAP) <= GAP_TOLERANCE:
        s['prune_p'] = p; s['A'] = A
        hits.append(s)
    if len(hits) >= GRAPHS_PER_FAMILY * 6: break

print(f'{len(hits)} graphs found inside the gap window after {it+1} draws.')
if len(hits) < 2 * GRAPHS_PER_FAMILY:
    print('NOT ENOUGH. Widen GAP_TOLERANCE or change TARGET_GAP, then rerun.')
else:
    H = pd.DataFrame(hits).sort_values('exc').reset_index(drop=True)
    LOW  = H.iloc[:GRAPHS_PER_FAMILY]
    HIGH = H.iloc[-GRAPHS_PER_FAMILY:]
    print()
    print(f'{"":<14}{"spectral gap":>16}{"triadic excess":>18}{"clustering":>14}{"mean degree":>14}')
    for nm, F in (("LOW family", LOW), ("HIGH family", HIGH)):
        print(f'{nm:<14}{F.gap.mean():>10.3f} +/-{F.gap.std():<5.3f}'
              f'{F.exc.mean():>13.4f} +/-{F.exc.std():<5.3f}'
              f'{F.clustering.mean():>10.3f}{F.mean_degree.mean():>14.1f}')
    print()
    print(f'gap difference between families:     {abs(HIGH.gap.mean()-LOW.gap.mean()):.4f}')
    print(f'triadic excess difference:           {abs(HIGH.exc.mean()-LOW.exc.mean()):.4f}')
    print(f'  ratio (excess sep / gap sep):      '
          f'{abs(HIGH.exc.mean()-LOW.exc.mean())/max(abs(HIGH.gap.mean()-LOW.gap.mean()),1e-9):.1f}x')

In [ ]:
#@title 4. Why not the clustering coefficient (reproduces the 62% / 19% figures)
rng2 = np.random.default_rng(42)
rows = []
for _ in range(300):
    p = rng2.uniform(0.05, 0.85)
    G = nx.random_regular_graph(K_REGULAR, N_NODES, seed=int(rng2.integers(1e9)))
    G.remove_edges_from([e for e in list(G.edges()) if rng2.random() < p])
    if G.number_of_edges() == 0: continue
    G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    if G.number_of_nodes() < N_NODES * 0.5: continue
    s, _ = summarise(nx.convert_node_labels_to_integers(G))
    rows.append(s)
D = pd.DataFrame(rows)
D['band'] = pd.qcut(D.gap, 12, labels=False, duplicates='drop')
print(f'spectral gap ranges from {D.gap.min():.2f} to {D.gap.max():.2f} '
      f'under pruning p in [0.05, 0.85]\n')
print(f'{"quantity":<22}{"total sd":>12}{"sd within gap band":>22}{"% free of the gap":>20}')
for col, lab in (('exc', 'triadic excess'), ('clustering', 'clustering coeff')):
    tot = D[col].std(); within = D.groupby('band')[col].std().mean()
    print(f'{lab:<22}{tot:>12.4f}{within:>22.4f}{100*within/max(tot,EPS):>19.1f}%')
print('\nThe clustering coefficient is largely pinned by the gap, so matching on')
print('the gap nearly matches on clustering too. The triadic excess is not.')

In [ ]:
#@title 5. Look at it
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(D.gap, D.exc, s=10, alpha=.5, label='triadic excess')
ax[0].scatter(D.gap, D.clustering, s=10, alpha=.5, label='clustering')
ax[0].set_xlabel('spectral gap'); ax[0].set_ylabel('local quantity')
ax[0].legend(); ax[0].set_title('what the gap does and does not pin down')
if len(hits) >= 2 * GRAPHS_PER_FAMILY:
    ax[1].scatter(LOW.gap, LOW.exc, s=40, label='LOW family')
    ax[1].scatter(HIGH.gap, HIGH.exc, s=40, label='HIGH family')
    ax[1].axvline(TARGET_GAP, ls='--', c='k', lw=1)
    ax[1].set_xlabel('spectral gap'); ax[1].set_ylabel('median triadic excess')
    ax[1].legend(); ax[1].set_title('the two matched families')
plt.tight_layout(); plt.show()

In [ ]:
#@title 6. Export the connectivity matrices
from google.colab import files
import scipy.io as sio
if len(hits) >= 2 * GRAPHS_PER_FAMILY:
    out = {'LOW': np.stack(LOW.A.values), 'HIGH': np.stack(HIGH.A.values),
           'target_gap': TARGET_GAP, 'tolerance': GAP_TOLERANCE,
           'gap_LOW': LOW.gap.values, 'gap_HIGH': HIGH.gap.values,
           'excess_LOW': LOW.exc.values, 'excess_HIGH': HIGH.exc.values}
    sio.savemat('gap_matched_families.mat', out)
    meta = pd.concat([LOW.assign(family='LOW'), HIGH.assign(family='HIGH')])
    meta.drop(columns=['A']).to_csv('gap_matched_families.csv', index=False)
    print('written: gap_matched_families.mat (Matlab) and .csv (summary table)')
    print('The .mat holds two arrays of shape (graphs, nodes, nodes), ready to')
    print('drop in wherever the pruned k-regular matrices currently go.')
    # files.download('gap_matched_families.mat')
else:
    print('Nothing to export yet: widen the tolerance in cell 2 and rerun.')

## What this does not settle

It produces the stimulus set, not the answer. Whether model fit changes when the
gap is held fixed has to be measured inside the whole-brain model, with the
empirical data and the interference measure, and that part is not here.

Both outcomes are informative. If fit changes at fixed gap, the gap was not what
drove it. If fit does not change, the original interpretation survives the
objection raised in the paper's own limitations, which is worth reporting.

The 62% / 19% split in cell 4 was measured on 600 graphs at n=40, k=20. It has
not been checked at other sizes, and cell 4 lets you rerun it wherever you
intend to work.